# 第 42 章：Capstone 案例模板与评分 Rubric

这个 notebook 用一个极小的 rubric case table，对比“只按总分排序”的 naive baseline 和“先检查 AI-use / signoff gate、核心维度最低线与 revision burden”的评分前 workflow。

In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_rubric_case_template_demo.csv')

POINT_FIELDS = [
    'science_question_points',
    'data_repro_points',
    'modeling_eval_points',
    'interpretation_limits_points',
    'code_git_points',
    'report_presentation_points',
]


def load_cases(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            for field in POINT_FIELDS:
                row[field] = float(row[field])
            row['revision_burden_score'] = float(row['revision_burden_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


def total_score(row):
    return sum(row[field] for field in POINT_FIELDS)


rows = load_cases(DATA_PATH)
print(f'Loaded {len(rows)} rubric case cards from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('AI-use integrity status:', ordered_counts(rows, 'ai_use_integrity_status'))
print('Signoff status:', ordered_counts(rows, 'signoff_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_ready_ids = sorted(
    row['case_id']
    for row in rows
    if row['reference_route'] == 'ready_for_grading'
)

top3_total = sorted(rows, key=total_score, reverse=True)[:3]
baseline_ready_hits = sum(
    row['reference_route'] == 'ready_for_grading'
    for row in top3_total
)
baseline_blocked_hits = sum(
    row['reference_route'] == 'blocked_before_grading'
    for row in top3_total
)
baseline_ready_precision = baseline_ready_hits / len(top3_total)
baseline_blocked_intrusion = baseline_blocked_hits / len(top3_total)

print('Reference ready-for-grading cases:', reference_ready_ids)
print('Top-3 by total score:')
for row in top3_total:
    print(
        f"  {row['case_id']}: total={total_score(row):.1f}, "
        f"route={row['reference_route']}, theme={row['theme']}"
    )
print('Baseline ready precision:', round(baseline_ready_precision, 3))
print('Baseline blocked intrusion:', round(baseline_blocked_intrusion, 3))


In [ ]:
def route_case(row):
    if row['signoff_status'] != 'signed':
        return 'blocked_before_grading', 'human signoff is not yet complete'
    if row['ai_use_integrity_status'] != 'clear':
        return 'blocked_before_grading', 'AI-use disclosure or integrity status is not clear'

    ready_floors = [
        row['science_question_points'] >= 12,
        row['data_repro_points'] >= 16,
        row['modeling_eval_points'] >= 20,
        row['interpretation_limits_points'] >= 16,
        row['code_git_points'] >= 8,
        row['report_presentation_points'] >= 8,
    ]
    minor_floors = [
        row['science_question_points'] >= 10,
        row['data_repro_points'] >= 12,
        row['modeling_eval_points'] >= 16,
        row['interpretation_limits_points'] >= 12,
        row['code_git_points'] >= 6,
        row['report_presentation_points'] >= 6,
    ]
    total = total_score(row)
    burden = row['revision_burden_score']

    if total >= 85 and all(ready_floors) and burden <= 0.25:
        return 'ready_for_grading', 'all major evidence gates and ready floors pass'
    if total >= 75 and all(minor_floors) and burden <= 0.40:
        return 'strong_with_minor_revision', 'overall project is strong but one or more ready floors still need polish'
    return 'needs_revision', 'score, core floors, or revision burden still require substantial work'


routed_rows = []
for row in rows:
    route, reason = route_case(row)
    routed = dict(row)
    routed['total_score'] = total_score(row)
    routed['route'] = route
    routed['reason'] = reason
    routed_rows.append(routed)

print('Rubric workflow routes:')
for row in routed_rows:
    print(
        f"  {row['case_id']}: total={row['total_score']:.1f}, "
        f"route={row['route']}, reason={row['reason']}"
    )


In [ ]:
ready_queue = [row for row in routed_rows if row['route'] == 'ready_for_grading']
minor_queue = [row for row in routed_rows if row['route'] == 'strong_with_minor_revision']
revision_queue = [row for row in routed_rows if row['route'] == 'needs_revision']
blocked_queue = [row for row in routed_rows if row['route'] == 'blocked_before_grading']

workflow_accuracy = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
) / len(routed_rows)
ready_precision = sum(
    row['reference_route'] == 'ready_for_grading'
    for row in ready_queue
) / max(len(ready_queue), 1)
blocked_intrusion = sum(
    row['reference_route'] == 'blocked_before_grading'
    for row in ready_queue
) / max(len(ready_queue), 1)

print('Ready-for-grading queue:', [row['case_id'] for row in ready_queue])
print('Minor-revision queue:', [row['case_id'] for row in minor_queue])
print('Needs-revision queue:', [row['case_id'] for row in revision_queue])
print('Blocked-before-grading queue:', [row['case_id'] for row in blocked_queue])
print('Workflow route accuracy:', round(workflow_accuracy, 3))
print('Structured ready precision:', round(ready_precision, 3))
print('Structured blocked intrusion:', round(blocked_intrusion, 3))


In [ ]:
def rubric_feedback(row):
    steps = []
    if row['signoff_status'] != 'signed':
        steps.append('先完成 human signoff、数据公开边界和最终提交责任确认。')
    if row['ai_use_integrity_status'] != 'clear':
        steps.append('补全 AI-use statement，并核查是否有未披露、不可验证或越界使用。')
    if row['science_question_points'] < 12:
        steps.append('重写科学问题，让输入数据、成功标准和可回答范围更清楚。')
    if row['data_repro_points'] < 16:
        steps.append('补数据来源、许可证、字段说明、清洗规则和可复跑划分。')
    if row['modeling_eval_points'] < 20:
        steps.append('补 baseline、validation/test 逻辑、失败样本和过拟合检查。')
    if row['interpretation_limits_points'] < 16:
        steps.append('补结果解释、局限性、不确定度和至少一个反例或失败样本。')
    if row['code_git_points'] < 8:
        steps.append('清理 notebook、固定随机种子、补 README，并保留有意义的 Git 记录。')
    if row['report_presentation_points'] < 8:
        steps.append('压缩报告和展示，让 2 到 3 张核心图支撑主要 claim。')
    if row['revision_burden_score'] > 0.40:
        steps.append('把修订拆成一周内可完成的小任务，不要一次性重做整个项目。')
    return steps or ['可以进入评分；评分者仍需阅读报告、复跑关键 notebook 并核查引用。']


def case_template_package(row):
    return {
        'project_card': row['theme'],
        'total_score': round(row['total_score'], 1),
        'route': row['route'],
        'reason': row['reason'],
        'revision_checklist': rubric_feedback(row),
    }


for row in routed_rows:
    if row['route'] != 'ready_for_grading':
        package = case_template_package(row)
        print(f"\n{row['case_id']} -> {package['route']} ({package['total_score']} pts)")
        for step in package['revision_checklist']:
            print(' -', step)


In [ ]:
rubric_assistant_prompt = '''你是我的 capstone rubric 助教。

任务：
1. 先读取 case template cards，不要只按总分排序；
2. 先检查 AI-use integrity 和 human signoff gate；
3. 再检查 science question、data reproducibility、modeling/evaluation、
   interpretation/limitations、code/Git、report/presentation 的最低线；
4. 把每个项目 route 到 ready_for_grading、strong_with_minor_revision、
   needs_revision 或 blocked_before_grading；
5. 对每个非 ready 项目输出 3 到 6 条具体 revision action。

输出格式：
- Ready-for-grading cases
- Minor-revision cases
- Needs-revision cases
- Blocked-before-grading cases
- Revision checklist by case
'''
print(rubric_assistant_prompt)


In [ ]:
case_template_cards = [
    'Project card: science question and success criterion',
    'Data card: source, license, fields, split, and cleaning rules',
    'Baseline card: simple baseline before model escalation',
    'Model card: model choice, inputs, outputs, and failure cases',
    'Evaluation card: metrics, validation/test split, and leakage checks',
    'Interpretation card: key claims, figures, limitations, and uncertainty',
    'Reproducibility card: notebook rerun, environment, seeds, and Git log',
    'Integrity card: AI-use statement, citation checks, signoff, and sharing boundary',
]

rubric_snapshot = {
    'dataset': DATA_PATH.name,
    'n_case_cards': len(rows),
    'baseline_top3_ready_precision': round(baseline_ready_precision, 3),
    'baseline_top3_blocked_intrusion': round(baseline_blocked_intrusion, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'ready_for_grading': [row['case_id'] for row in ready_queue],
    'minor_revision': [row['case_id'] for row in minor_queue],
    'blocked_before_grading': [row['case_id'] for row in blocked_queue],
    'python_version': platform.python_version(),
}

print('Case-template cards:')
for card in case_template_cards:
    print(' -', card)

print('\nRubric snapshot:')
for key, value in rubric_snapshot.items():
    print(f'  {key}: {value}')
